# Dunnhumby seed 43: M4 → M5 Top-10 정답 유입·이탈 진단

수정 원형 M4 λ=0.25(선택 25 epoch)와 선형 N/V 결합 M5 λ=0.25(선택 150 epoch)의 **기존 체크포인트만** 사용합니다. 신규상품 개발구간의 전체·저/중/고CLV 사용자별 Top-10 이동과 가격·구매금액 가중 적중값의 유입·이탈 기여를 계산합니다. 11–20위, 21–50위, 50위 밖에서 온 정답을 구분합니다. 학습·optimizer·체크포인트 재선택·최종 test/holdout 평가는 없습니다.

동일 실행의 M1은 선택 체크포인트가 저장되지 않았습니다(`checkpoint: null`). M1 집계 지표는 출처 확인용으로만 기록하고 **M1 사용자별 순위 이동은 계산하지 않습니다**. M5 내부 ID-only 점수를 M1로 대체하지 않습니다. 반복 노출된 개발분할의 단일 시드 사후 진단이며 성능 귀속·유의성·일반화의 근거가 아닙니다.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from google.colab import drive
ROOT = Path('/content/drive/MyDrive/논문/data')
REPORT = ROOT/'results_v3_dunnhumby_m5_linear_nv_original_m4_lambda025_seed43_v1/reports/result.json'
if not REPORT.is_file():
    if os.path.ismount('/content/drive'):
        raise RuntimeError('Drive는 연결됐지만 정확한 M5 결과를 찾을 수 없습니다. 연결 계정과 REPORT 경로를 확인하세요.')
    try:
        drive.mount('/content/drive')
    except (ValueError, NotImplementedError) as exc:
        raise RuntimeError('Drive 연결 실패. 코드는 학습이나 평가를 시작하지 않았습니다.') from exc
    if not REPORT.is_file():
        raise RuntimeError('Drive 연결 후에도 정확한 M5 결과를 찾을 수 없습니다. REPORT 경로를 확인하세요.')
SOURCE_COMMIT = '70c771c8d25220e147eff359d1e04f13fd31da6f'
REPO = Path('/content/clv-top10-movement-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
if 'lightgcn_clv_v3' in sys.modules:
    assert Path(sys.modules['lightgcn_clv_v3'].__file__).resolve().parent == REPO.resolve(), '다른 소스가 로드됐습니다. 런타임을 재시작하세요.'
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import clv_m4_m5_top10_movement_diagnostic as diagnostic
import pandas as pd
OUT = ROOT/'results_v3_dunnhumby_m4_m5_top10_movement_seed43_v1'
print('진단 버전:', diagnostic.VERSION, '| 새 학습 없음')

## 1. 원본 결과 확인
원본 JSON의 SHA-256과 선택 체크포인트 파일을 검증합니다. 누락되거나 바뀌었으면 자동 재학습하지 않고 중단합니다.

In [ ]:
report, cfg = diagnostic._verify_report(REPORT)
for model_id in diagnostic.MODEL_IDS:
    arm = diagnostic._one_arm(report, model_id)
    print(model_id, '| 선택 epoch:', arm['selected_epoch'], '| checkpoint:', arm['checkpoint'])
print('M1 사용자별 순위 분석 불가: 저장된 선택 체크포인트 없음')

## 2. 저장된 체크포인트로 개발구간 추천목록 재생성
기존 집계 지표와 재계산 지표가 맞는지 확인한 뒤, 정답 상품의 Top-10 유입·이탈을 저장합니다. 이 단계는 GPU를 사용할 수 있지만 **학습은 하지 않습니다**.

In [ ]:
paths = diagnostic.run(REPORT, OUT)
summary = pd.read_csv(paths['summary'])
display(summary)
print(json.dumps(paths, ensure_ascii=False, indent=2))
print('해석: 선택 epoch가 다른 두 모델의 사후 순위 이동입니다. 원인·인과효과 또는 새 운영점의 성공 판정이 아닙니다.')

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
archive = OUT/'m4_m5_top10_movement_seed43.zip'
with ZipFile(archive, 'w', compression=ZIP_DEFLATED) as zipped:
    for path in paths.values():
        zipped.write(path, arcname=Path(path).name)
print('진단 ZIP:', archive)
files.download(str(archive))